# Hamiltonian simulation

Trotterize a transverse-field Ising chain and compare an observable trajectory.

The SDK reference and MettleQ calls below use the same circuit and result contract. Timing includes the complete call shown.

In [1]:
import numpy as np
from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import SparsePauliOp, Statevector
from qiskit.primitives import StatevectorEstimator, StatevectorSampler

from mettleq.integrations.qiskit import (
    MettleQBackend,
    MettleQEstimatorV2,
    MettleQSamplerV2,
)
from tutorials._support import (
    benchmark,
    emit_result,
    max_abs_error,
    phase_aligned_statevector_error,
    qiskit_selection,
    total_variation_distance,
)

In [2]:
times = np.linspace(0.0, 1.2, 13)
observable = SparsePauliOp("IIZ")

def evolution_circuit(time_value):
    circuit = QuantumCircuit(3)
    circuit.x(0)
    steps = 6
    dt = float(time_value) / steps
    for _ in range(steps):
        circuit.rzz(1.1 * dt, 0, 1)
        circuit.rzz(1.1 * dt, 1, 2)
        for wire in range(3):
            circuit.rx(0.7 * dt, wire)
    return circuit

circuits = [evolution_circuit(value) for value in times]

def reference_trajectory():
    estimator = StatevectorEstimator()
    return np.asarray([estimator.run([(c, observable)]).result()[0].data.evs.item() for c in circuits])

reference, reference_ms, _ = benchmark(reference_trajectory)
backend = MettleQBackend(method="statevector", device="cpu")
compiled = [transpile(c, backend, optimization_level=1) for c in circuits]
estimator = MettleQEstimatorV2(backend=backend)

def mettleq_trajectory():
    return np.asarray([estimator.run([(c, observable)]).result()[0].data.evs.item() for c in compiled])

candidate, mettleq_ms, _ = benchmark(mettleq_trajectory)
error = max_abs_error(reference, candidate)
method, device = qiskit_selection(estimator)
tutorial_result = emit_result(
    notebook="qiskit/12_hamiltonian_simulation.ipynb",
    framework="qiskit",
    reference_ms=reference_ms,
    mettleq_ms=mettleq_ms,
    check="observable trajectory atol=3e-6",
    passed=error <= 3e-6,
    exact_match=bool(np.array_equal(reference, candidate)),
    selected_method=method,
    selected_device=device,
    metrics={"max_observable_error": error, "times": times, "reference": reference, "mettleq": candidate},
)

TUTORIAL_RESULT::{"check": "observable trajectory atol=3e-6", "exact_match": false, "framework": "qiskit", "machine": "arm64", "metrics": {"max_observable_error": 1.2218022292787012e-06, "mettleq": [-1.0, -0.997553825378418, -0.9902541637420654, -0.9782247543334961, -0.9616613984107971, -0.9408406019210815, -0.9161059856414795, -0.8878596425056458, -0.8565607666969299, -0.8227137923240662, -0.7868613600730896, -0.7495623230934143, -0.71140056848526], "reference": [-1.0, -0.9975533999646712, -0.9902542917948219, -0.9782239468098275, -0.9616618196615209, -0.9408416125730155, -0.9161058849705573, -0.8878593249231733, -0.8565608274635523, -0.8227145502198748, -0.7868601382708603, -0.7495623272636097, -0.7114001462281647], "times": [0.0, 0.09999999999999999, 0.19999999999999998, 0.3, 0.39999999999999997, 0.49999999999999994, 0.6, 0.7, 0.7999999999999999, 0.8999999999999999, 0.9999999999999999, 1.0999999999999999, 1.2]}, "mettleq_median_ms": 40.46845802804455, "notebook": "qiskit/12_hamilton